# Incident 5 — The Category Violation

**What happened:** A travel booking agent was authorized for flights ($600) and hotels ($400). The agent tried to book a hotel but mistakenly charged it to the flight allocation. Without enforcement it went through silently. The flight budget was depleted for non-flight spend. Finance caught it in the **month-end review**.

**Why it happens:** Budget limits stop overspend but don't enforce *what* the money is spent on. An agent can drain the flight allocation with hotel charges — the total stays within limits, but the intent is violated.

**The fix:** `CATEGORY_CONSTRAINED` allocations. FiGuard denies any spend whose `claimed_category` doesn't match the allocation's `allowed_categories`. The flight budget can only be used for flight spend, hotel budget for hotel spend — enforced at authorization time, not discovered at month-end.

In [ ]:
import sys
!pip install "figuard>=0.3.2" --upgrade -q
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} ready")

## Without FiGuard — hotel charged to flight allocation silently

The total spend stays within the $1,000 budget. The miscategorization passes through — the flight allocation is drained by hotel spend.

In [ ]:
# WITHOUT FiGuard — no category enforcement

allocations = {
    "flight": {"limit": 600.00, "spent": 0.0},
    "hotel":  {"limit": 400.00, "spent": 0.0},
}

def charge_allocation(alloc_name, amount, actual_category):
    alloc = allocations[alloc_name]
    remaining = alloc["limit"] - alloc["spent"]
    if amount <= remaining:
        alloc["spent"] += amount
        return "AUTHORIZED"
    return "DENIED"

print(f"Budget: $1,000.00  |  flight $600  hotel $400")
print()

# Correct: flight booked to flight allocation
r1 = charge_allocation("flight", 267.00, actual_category="flight")
print(f"Flight → flight alloc:  {r1} — $267.00")

# Bug: hotel booked to flight allocation (agent picks wrong category)
r2 = charge_allocation("flight", 312.00, actual_category="hotel")
print(f"Hotel  → flight alloc:  {r2} — $312.00  ← WRONG, but passes through")

# Correct hotel booking — but now flight allocation is partially drained by hotel spend
r3 = charge_allocation("hotel", 312.00, actual_category="hotel")
print(f"Hotel  → hotel alloc:   {r3} — $312.00")

print()
for name, alloc in allocations.items():
    print(f"  {name:6s} allocation: ${alloc['spent']:.2f} spent of ${alloc['limit']:.2f}")
print()
print("Flight allocation contaminated: hotel spend passed as flight — discovered at month-end")

## With FiGuard — category mismatch denied at authorization time

`CATEGORY_CONSTRAINED` allocations reject any spend whose `claimed_category` doesn't match `allowed_categories`. The hotel charge against the flight allocation is denied immediately — the agent is forced to use the correct allocation.

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://figuard-sandbox-1.onrender.com",
)

budget = client.create_budget(
    user_id="travel_agent",
    total_limit=1000.00,
    currency="USD",
    expires_in="1h",
    allocations=[
        {
            "category": "flight",
            "limit": 600.00,
            "enforcement_mode": "CATEGORY_CONSTRAINED",
            "allowed_categories": ["flight"],
        },
        {
            "category": "hotel",
            "limit": 400.00,
            "enforcement_mode": "CATEGORY_CONSTRAINED",
            "allowed_categories": ["hotel"],
        },
    ],
)

# Extract session token from tokens array (one entry per dimension)
tokens = {t.category: t.session_token for t in budget.tokens}
session_token = tokens["default"]  # simple budget uses "default" category


print(f"Budget: ${budget.total_limit:.2f}  |  flight $600  hotel $400")
print()

# Correct: flight booked against flight allocation
auth1 = client.authorize(
    session_token=session_token,
    agent_id="travel_agent",
    action_type="PURCHASE",
    description="JetBlue SFO→JFK",
    requested_quantity=267.00,
    claimed_category="flight",
    idempotency_key="booking-flight-001",
)
print(f"Flight → flight:  {auth1.decision} — $267.00")
if auth1.is_authorized:
    client.confirm_event(auth1.event_id, confirmed_quantity=267.00)

# Wrong: hotel charged to flight allocation
auth2 = client.authorize(
    session_token=session_token,
    agent_id="travel_agent",
    action_type="PURCHASE",
    description="Marriott Times Square",
    requested_quantity=312.00,
    claimed_category="flight",   # wrong category
    idempotency_key="booking-hotel-wrong",
)
print(f"Hotel  → flight:  {auth2.decision}"
      + (f" — {auth2.denial_reason}" if not auth2.is_authorized else " — $312.00"))

# Correct: hotel charged to hotel allocation
auth3 = client.authorize(
    session_token=session_token,
    agent_id="travel_agent",
    action_type="PURCHASE",
    description="Marriott Times Square",
    requested_quantity=312.00,
    claimed_category="hotel",    # correct
    idempotency_key="booking-hotel-001",
)
print(f"Hotel  → hotel:   {auth3.decision} — $312.00")
if auth3.is_authorized:
    client.confirm_event(auth3.event_id, confirmed_quantity=312.00)

print()
print("✓ Flight allocation: $267.00 (flights only)")
print("  Hotel allocation:  $312.00 (hotels only)")
print("  Category contamination blocked at authorization time.")

print(f"\n→ See the ledger: https://figuard-sandbox-1.onrender.com/ui")